# Poisson Equation PINTO Model - Post Processing

This notebook provides comprehensive post-processing analysis for the 1D Poisson equation PINTO model, including visualization, performance metrics, and comparison with numerical solutions.

## Import Required Libraries

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as tick
import matplotlib as mpl
import seaborn as sns
import h5py
import sys
sys.path.append('../Code')
sys.path.append('.')
from utils import read_h5_file, prepare_prediction_data, compute_error_metrics, load_training_history

get_dir = os.getcwd()
os.chdir(get_dir)

## Load Prediction Functions

In [ ]:
def get_predictions_and_true_data_poisson(mdir, ddir, indices, context_dir):
    """
    Generate predictions for Poisson equation using trained PINTO model.
    
    Args:
        mdir: Model directory path
        ddir: Data directory path (HDF5 file)
        indices: List of sample indices to process
        context_dir: Directory containing context data (CSV file)
    
    Returns:
        xdisc: Spatial discretization
        p_pred: Predicted solutions
        p_true: True solutions
        a_coeff: Coefficients
        f_force: Forcing terms
    """
    try:
        context_indices = pd.read_csv(context_dir)['context_indices'].values.reshape(-1)
    except:
        print("Context file not found, using default context indices")
        context_indices = np.arange(60)  # Default context size
    
    print(f"Context indices shape: {context_indices.shape}")
    model = tf.keras.models.load_model(mdir, safe_mode=False)
    
    # Read Poisson data using the new utility function
    p_all, a_all, f_all, bc_all, xdisc, L = read_h5_file(ddir)
    
    p_pred = []
    p_true = []
    a_coeff = []
    f_force = []
    
    for i in indices:
        print(f"Processing sample {i}...")
        
        # Prepare data for this sample using utility function
        sample_data = prepare_prediction_data(p_all, a_all, f_all, xdisc, context_indices, i)
        
        # Get inputs for model prediction
        x_input = sample_data['x']
        a_input = sample_data['a']
        a_context = sample_data['a_context']
        f_context = sample_data['f_context']
        f_values = sample_data['f_context']  # Using f_context as f_values
        
        # Ensure proper shapes for model input
        x_input = x_input.reshape(-1, 1)
        a_input = a_input.reshape(-1, 1)
        a_context = a_context.reshape(len(x_input), -1, 1)
        f_context = f_context.reshape(len(x_input), -1, 1)
        f_values = f_values.reshape(len(x_input), -1, 1)
        
        # Generate predictions
        p_prediction = model.predict([x_input, a_input, a_context, f_context, f_values], 
                                   batch_size=1024, verbose=0)
        
        p_pred.append(p_prediction.flatten())
        p_true.append(sample_data['true_solution'])
        a_coeff.append(a_all[i])
        f_force.append(f_all[i])
    
    print("Prediction generation completed!")
    return xdisc, p_pred, p_true, a_coeff, f_force

## Load Plotting Functions

In [ ]:
def plot_poisson_solutions(xdisc, p_pred, p_true, a_coeff, f_force, indices, 
                          plot_save=False, plot_dir='poisson_results', fig_size=(15, 10)):
    """
    Create comprehensive plots for Poisson equation solutions.
    
    Args:
        xdisc: Spatial discretization
        p_pred: Predicted solutions
        p_true: True solutions  
        a_coeff: Coefficient functions
        f_force: Forcing terms
        indices: Sample indices being plotted
        plot_save: Whether to save the plot
        plot_dir: Directory to save plots
        fig_size: Figure size tuple
    """
    n_samples = len(p_pred)
    fig, ax = plt.subplots(4, n_samples, figsize=fig_size, sharex=True)
    
    if n_samples == 1:
        ax = ax.reshape(-1, 1)
    
    for i in range(n_samples):
        # Solution p(x)
        ax[0][i].plot(xdisc, p_pred[i], 'b-', label='PINTO Prediction', linewidth=2)
        ax[0][i].plot(xdisc, p_true[i], 'r--', label='True Solution', linewidth=2)
        ax[0][i].set_title(f'Solution p(x) - Sample {indices[i]}')
        ax[0][i].set_ylabel('p(x)')
        ax[0][i].legend()
        ax[0][i].grid(True, alpha=0.3)
        
        # Coefficient a(x)
        ax[1][i].plot(xdisc, a_coeff[i], 'g-', label='Coefficient a(x)', linewidth=2)
        ax[1][i].set_title(f'Coefficient a(x) - Sample {indices[i]}')
        ax[1][i].set_ylabel('a(x)')
        ax[1][i].legend()
        ax[1][i].grid(True, alpha=0.3)
        
        # Forcing term f(x)
        ax[2][i].plot(xdisc, f_force[i], 'm-', label='Forcing f(x)', linewidth=2)
        ax[2][i].set_title(f'Forcing Term f(x) - Sample {indices[i]}')
        ax[2][i].set_ylabel('f(x)')
        ax[2][i].legend()
        ax[2][i].grid(True, alpha=0.3)
        
        # Relative error
        rel_error = np.abs(p_pred[i] - p_true[i]) / (1 + np.abs(p_true[i]))
        ax[3][i].plot(xdisc, rel_error, 'k-', label='Relative Error', linewidth=2)
        ax[3][i].set_title(f'Relative Error - Sample {indices[i]}')
        ax[3][i].set_ylabel('Relative Error')
        ax[3][i].set_xlabel('x')
        ax[3][i].legend()
        ax[3][i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if plot_save:
        plt.savefig(f'{plot_dir}.png', dpi=300, bbox_inches='tight', format='png')
    
    plt.show()


def plot_solution_comparison(xdisc, p_pred, p_true, a_coeff, indices, 
                           plot_save=False, plot_dir='comparison', fig_size=(12, 8)):
    """
    Create side-by-side comparison plots of solutions with coefficient overlay.
    """
    n_samples = len(p_pred)
    fig, ax = plt.subplots(2, n_samples, figsize=fig_size, sharex=True)
    
    if n_samples == 1:
        ax = ax.reshape(-1, 1)
    
    for i in range(n_samples):
        # Main solution comparison
        ax[0][i].plot(xdisc, p_pred[i], 'b-', label='PINTO Prediction', linewidth=2.5)
        ax[0][i].plot(xdisc, p_true[i], 'r--', label='True Solution', linewidth=2.5)
        ax[0][i].set_title(f'Solution Comparison - Sample {indices[i]}', fontweight='bold')
        ax[0][i].set_ylabel('p(x)', fontweight='bold')
        ax[0][i].legend(frameon=False, fontsize=10)
        ax[0][i].grid(True, alpha=0.3)
        
        # Error analysis with coefficient background
        ax2 = ax[1][i].twinx()
        error = p_pred[i] - p_true[i]
        ax[1][i].plot(xdisc, error, 'k-', label='Absolute Error', linewidth=2)
        ax2.fill_between(xdisc, a_coeff[i], alpha=0.3, color='lightblue', label='Coefficient a(x)')
        
        ax[1][i].set_title(f'Error Analysis - Sample {indices[i]}', fontweight='bold')
        ax[1][i].set_xlabel('x', fontweight='bold')
        ax[1][i].set_ylabel('Absolute Error', fontweight='bold')
        ax2.set_ylabel('Coefficient a(x)', fontweight='bold')
        
        # Combined legend
        lines1, labels1 = ax[1][i].get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax[1][i].legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=9)
        ax[1][i].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if plot_save:
        plt.savefig(f'{plot_dir}.png', dpi=300, bbox_inches='tight', format='png')
    
    plt.show()

## Load Trained Model and Generate Predictions

In [ ]:
# Set up paths and parameters
mdir = '../Code/output/Poisson_PINTO/Poisson_model'  # Path to trained model
ddir = '../Code/dataset/poisson_1d.h5'  # Path to Poisson data
context_dir = '../Code/output/Poisson_PINTO/context.csv'  # Path to context indices

# Define sample indices for analysis
train_indices = [10, 25, 40, 55]  # Seen boundary conditions
test_indices = [85, 90, 95]       # Unseen boundary conditions
all_indices = train_indices + test_indices

print("Loading trained Poisson PINTO model and generating predictions...")
print(f"Model directory: {mdir}")
print(f"Data directory: {ddir}")
print(f"Context directory: {context_dir}")
print(f"Analyzing samples: {all_indices}")

# Generate predictions
xdisc, p_pred, p_true, a_coeff, f_force = get_predictions_and_true_data_poisson(
    mdir, ddir, all_indices, context_dir
)

print(f"Spatial discretization: {len(xdisc)} points from {xdisc[0]:.3f} to {xdisc[-1]:.3f}")
print(f"Generated predictions for {len(p_pred)} samples")
print("Prediction generation completed!")

## Create Solution Plots

In [ ]:
# Plot comprehensive solution analysis for training samples
print("Creating plots for training samples (seen boundary conditions)...")
plot_poisson_solutions(xdisc, p_pred[:len(train_indices)], p_true[:len(train_indices)], 
                       a_coeff[:len(train_indices)], f_force[:len(train_indices)], 
                       train_indices, plot_save=True, plot_dir='Poisson_Train_Results', 
                       fig_size=(16, 12))

# Plot comprehensive solution analysis for test samples  
print("Creating plots for test samples (unseen boundary conditions)...")
plot_poisson_solutions(xdisc, p_pred[len(train_indices):], p_true[len(train_indices):], 
                       a_coeff[len(train_indices):], f_force[len(train_indices):], 
                       test_indices, plot_save=True, plot_dir='Poisson_Test_Results', 
                       fig_size=(12, 12))

## Create Detailed Comparison Plots

In [ ]:
# Create detailed comparison plots
print("Creating detailed comparison plots...")

# Select representative samples for detailed analysis
representative_indices = [train_indices[1], test_indices[1]]  # One from each group
rep_pred = [p_pred[1], p_pred[len(train_indices) + 1]]
rep_true = [p_true[1], p_true[len(train_indices) + 1]]
rep_coeff = [a_coeff[1], a_coeff[len(train_indices) + 1]]

plot_solution_comparison(xdisc, rep_pred, rep_true, rep_coeff, 
                        representative_indices, plot_save=True, 
                        plot_dir='Poisson_Detailed_Comparison', fig_size=(14, 8))

## Generate Performance Metrics

In [ ]:
# Calculate metrics using the new utility functions
print("Calculating performance metrics...")
print("="*50)

# Calculate sample-wise metrics using the utility function
sample_errors = []
l2_errors = []
max_errors = []

for i, (pred, true, idx) in enumerate(zip(p_pred, p_true, all_indices)):
    # Use the utility function to compute comprehensive metrics
    metrics = compute_error_metrics(pred, true)
    
    sample_type = "Training" if i < len(train_indices) else "Test"
    sample_errors.append({
        'Sample_Index': idx,
        'Type': sample_type,
        'L2_Error': metrics['rel_error_l2'],
        'Max_Error': metrics['max_error'],
        'MAE': metrics['mae'],
        'RMSE': metrics['rmse']
    })
    
    l2_errors.append(metrics['rel_error_l2'])
    max_errors.append(metrics['max_error'])
    
    print(f"Sample {idx:2d} ({sample_type:8s}): L2={metrics['rel_error_l2']:.6f}, Max={metrics['max_error']:.6f}")

# Convert to DataFrame for analysis
sample_errors_df = pd.DataFrame(sample_errors)

# Calculate aggregate metrics
train_mask = sample_errors_df['Type'] == 'Training'
test_mask = sample_errors_df['Type'] == 'Test'

train_metrics = {
    'Mean_L2_Error': sample_errors_df[train_mask]['L2_Error'].mean(),
    'Mean_Max_Error': sample_errors_df[train_mask]['Max_Error'].mean(),
    'Mean_MAE': sample_errors_df[train_mask]['MAE'].mean(),
    'Mean_RMSE': sample_errors_df[train_mask]['RMSE'].mean(),
    'Std_L2_Error': sample_errors_df[train_mask]['L2_Error'].std(),
    'Std_Max_Error': sample_errors_df[train_mask]['Max_Error'].std()
}

test_metrics = {
    'Mean_L2_Error': sample_errors_df[test_mask]['L2_Error'].mean(),
    'Mean_Max_Error': sample_errors_df[test_mask]['Max_Error'].mean(),
    'Mean_MAE': sample_errors_df[test_mask]['MAE'].mean(),
    'Mean_RMSE': sample_errors_df[test_mask]['RMSE'].mean(),
    'Std_L2_Error': sample_errors_df[test_mask]['L2_Error'].std(),
    'Std_Max_Error': sample_errors_df[test_mask]['Max_Error'].std()
}

# Create comprehensive metrics table
metrics_df = pd.DataFrame({
    'Training Samples': [train_metrics[key] for key in train_metrics.keys()],
    'Test Samples': [test_metrics[key] for key in test_metrics.keys()]
}, index=list(train_metrics.keys()))

print("\nPerformance Metrics for Poisson PINTO Model:")
print(metrics_df.round(6))

print(f"\nMean Training L2 Error: {train_metrics['Mean_L2_Error']:.6f} ± {train_metrics['Std_L2_Error']:.6f}")
print(f"Mean Test L2 Error:     {test_metrics['Mean_L2_Error']:.6f} ± {test_metrics['Std_L2_Error']:.6f}")

# Create error distribution plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# L2 Error distribution
train_l2 = sample_errors_df[train_mask]['L2_Error']
test_l2 = sample_errors_df[test_mask]['L2_Error']

ax1.hist(train_l2, bins=10, alpha=0.7, label='Training', color='blue', density=True)
ax1.hist(test_l2, bins=10, alpha=0.7, label='Test', color='red', density=True)
ax1.set_xlabel('L2 Relative Error')
ax1.set_ylabel('Density')
ax1.set_title('L2 Error Distribution')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Sample-wise error comparison
sample_indices = sample_errors_df['Sample_Index']
l2_values = sample_errors_df['L2_Error']
colors = ['blue' if t == 'Training' else 'red' for t in sample_errors_df['Type']]

ax2.scatter(sample_indices, l2_values, c=colors, alpha=0.7)
ax2.set_xlabel('Sample Index')
ax2.set_ylabel('L2 Relative Error')
ax2.set_title('Error vs Sample Index')
ax2.grid(True, alpha=0.3)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='blue', label='Training'),
                  Patch(facecolor='red', label='Test')]
ax2.legend(handles=legend_elements)

plt.tight_layout()
plt.savefig('Poisson_Error_Analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## Create Learning Curves Visualization

In [ ]:
# Plot training history and learning curves using utility function
try:
    print("Loading and plotting training history...")
    
    # Load training history using utility function
    history_result = load_training_history('../Code/output/Poisson_PINTO/')
    
    if history_result['success']:
        history_data = history_result['train']
        val_history_data = history_result['val']
        
        # Set up the plot style
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Total Loss
        axes[0, 0].plot(history_data.index, np.log(history_data['loss']), 'b-', label='Training Loss', linewidth=2)
        if len(val_history_data) > 0:
            axes[0, 0].plot(val_history_data.index * (len(history_data) // len(val_history_data)), 
                            np.log(val_history_data['val_loss']), 'r--', label='Validation Loss', linewidth=2)
        axes[0, 0].set_title('Total Loss Evolution', fontweight='bold', fontsize=12)
        axes[0, 0].set_xlabel('Epochs')
        axes[0, 0].set_ylabel('Log Loss')
        axes[0, 0].legend()
        axes[0, 0].grid(True, alpha=0.3)
        
        # Residual Loss
        axes[0, 1].plot(history_data.index, np.log(history_data['residual_loss']), 'g-', 
                        label='Residual Loss', linewidth=2)
        if len(val_history_data) > 0 and 'val_res_loss' in val_history_data.columns:
            axes[0, 1].plot(val_history_data.index * (len(history_data) // len(val_history_data)), 
                            np.log(val_history_data['val_res_loss']), 'orange', linestyle='--', 
                            label='Val Residual Loss', linewidth=2)
        axes[0, 1].set_title('PDE Residual Loss Evolution', fontweight='bold', fontsize=12)
        axes[0, 1].set_xlabel('Epochs')
        axes[0, 1].set_ylabel('Log Residual Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True, alpha=0.3)
        
        # Boundary Loss
        if 'bound_loss' in history_data.columns:
            axes[1, 0].plot(history_data.index, np.log(history_data['bound_loss']), 'm-', 
                            label='Boundary Loss', linewidth=2)
            axes[1, 0].set_title('Boundary Condition Loss Evolution', fontweight='bold', fontsize=12)
            axes[1, 0].set_xlabel('Epochs')
            axes[1, 0].set_ylabel('Log Boundary Loss')
            axes[1, 0].legend()
            axes[1, 0].grid(True, alpha=0.3)
        else:
            axes[1, 0].text(0.5, 0.5, 'Boundary Loss\nNot Available', 
                           ha='center', va='center', transform=axes[1, 0].transAxes)
            axes[1, 0].set_title('Boundary Loss Not Available')
        
        # Data Loss (Validation)
        if len(val_history_data) > 0 and 'val_data_loss' in val_history_data.columns:
            axes[1, 1].plot(val_history_data.index * (len(history_data) // len(val_history_data)), 
                            np.log(val_history_data['val_data_loss']), 'c-', 
                            label='Validation Data Loss', linewidth=2)
            axes[1, 1].set_title('Validation Data Loss Evolution', fontweight='bold', fontsize=12)
            axes[1, 1].set_xlabel('Epochs')
            axes[1, 1].set_ylabel('Log Data Loss')
            axes[1, 1].legend()
            axes[1, 1].grid(True, alpha=0.3)
        else:
            axes[1, 1].text(0.5, 0.5, 'Validation Data Loss\nNot Available', 
                           ha='center', va='center', transform=axes[1, 1].transAxes)
            axes[1, 1].set_title('Validation Data Loss Not Available')
        
        plt.tight_layout()
        plt.savefig('Poisson_Learning_Curves.png', dpi=300, bbox_inches='tight')
        plt.show()
        
        # Print final training statistics
        print(f"\nFinal Training Statistics:")
        print(f"Final Training Loss: {history_data['loss'].iloc[-1]:.6f}")
        if len(val_history_data) > 0:
            print(f"Final Validation Loss: {val_history_data['val_loss'].iloc[-1]:.6f}")
        print(f"Final Residual Loss: {history_data['residual_loss'].iloc[-1]:.6f}")
        if 'bound_loss' in history_data.columns:
            print(f"Final Boundary Loss: {history_data['bound_loss'].iloc[-1]:.6f}")
        
    else:
        print(f"Could not load training history: {history_result['error']}")
        print("Skipping learning curves visualization...")
        
except Exception as e:
    print(f"Error in learning curves visualization: {e}")
    print("Please ensure the model has been trained and history files exist.")

## Computational Performance Analysis

In [ ]:
# Analyze computational performance and inference time
import time
import tensorflow as tf

def analyze_inference_performance(model, test_data_batch):
    """
    Analyze model inference performance
    """
    print("Analyzing computational performance...")
    
    # Warm up the model
    for _ in range(3):
        _ = model(test_data_batch)
    
    # Time inference
    num_runs = 50
    inference_times = []
    
    for _ in range(num_runs):
        start_time = time.time()
        predictions = model(test_data_batch)
        end_time = time.time()
        inference_times.append(end_time - start_time)
    
    avg_inference_time = np.mean(inference_times)
    std_inference_time = np.std(inference_times)
    
    print(f"Average inference time: {avg_inference_time*1000:.3f} ± {std_inference_time*1000:.3f} ms")
    print(f"Batch size: {test_data_batch[0].shape[0]}")
    print(f"Time per sample: {avg_inference_time*1000/test_data_batch[0].shape[0]:.3f} ms")
    
    return avg_inference_time, std_inference_time

def get_model_parameters(model):
    """
    Count model parameters
    """
    total_params = model.count_params()
    trainable_params = sum([tf.keras.backend.count_params(w) for w in model.trainable_weights])
    
    print(f"\\nModel Architecture Summary:")
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Non-trainable parameters: {total_params - trainable_params:,}")
    
    return total_params, trainable_params

# Performance analysis
if 'model' in locals():
    try:
        # Get a test batch for performance analysis
        test_batch_size = 32
        
        # Create a test batch from your test data
        if 'test_x_coords' in locals() and 'test_p_data' in locals():
            test_indices = np.random.choice(len(test_x_coords), test_batch_size, replace=False)
            
            test_coords_batch = test_x_coords[test_indices]
            test_p_batch = test_p_data[test_indices]
            test_a_batch = test_a_data[test_indices]
            test_f_batch = test_f_data[test_indices]
            
            # Create context data
            context_data = np.column_stack([test_p_batch.flatten(), 
                                          test_a_batch.flatten(), 
                                          test_f_batch.flatten()])
            context_data = context_data.reshape(test_batch_size, -1, 3)
            
            test_data_batch = [test_coords_batch, context_data]
            
            # Analyze performance
            avg_time, std_time = analyze_inference_performance(model, test_data_batch)
            total_params, trainable_params = get_model_parameters(model)
            
            # Create performance summary plot
            fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
            
            # Parameter distribution
            param_labels = ['Trainable', 'Non-trainable']
            param_values = [trainable_params, total_params - trainable_params]
            colors = ['skyblue', 'lightcoral']
            
            ax1.pie(param_values, labels=param_labels, colors=colors, autopct='%1.1f%%', startangle=90)
            ax1.set_title('Model Parameter Distribution', fontweight='bold')
            
            # Inference time visualization
            test_batch_sizes = [1, 8, 16, 32, 64, 128]
            estimated_times = [avg_time * bs / test_batch_size for bs in test_batch_sizes]
            
            ax2.plot(test_batch_sizes, estimated_times, 'o-', linewidth=2, markersize=6)
            ax2.set_xlabel('Batch Size')
            ax2.set_ylabel('Estimated Inference Time (s)')
            ax2.set_title('Inference Time vs Batch Size', fontweight='bold')
            ax2.grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.savefig('Poisson_Performance_Analysis.png', dpi=300, bbox_inches='tight')
            plt.show()
            
        else:
            print("Test data not available for performance analysis")
            get_model_parameters(model)
            
    except Exception as e:
        print(f"Error in performance analysis: {e}")
        if 'model' in locals():
            get_model_parameters(model)
else:
    print("Model not loaded. Please run the model loading cell first.")

## Summary and Conclusions

In [ ]:
# Generate comprehensive summary of results
print("="*80)
print("POISSON PINTO MODEL - COMPREHENSIVE ANALYSIS SUMMARY")
print("="*80)

try:
    # Summary statistics
    if 'l2_errors' in locals() and 'max_errors' in locals():
        print(f"\\n📊 ACCURACY METRICS:")
        print(f"   • Mean L2 Relative Error: {np.mean(l2_errors):.6f} ± {np.std(l2_errors):.6f}")
        print(f"   • Mean Maximum Error: {np.mean(max_errors):.6f} ± {np.std(max_errors):.6f}")
        print(f"   • Best Case L2 Error: {np.min(l2_errors):.6f}")
        print(f"   • Worst Case L2 Error: {np.max(l2_errors):.6f}")
        
        # Error distribution analysis
        excellent_cases = np.sum(l2_errors < 0.01)
        good_cases = np.sum((l2_errors >= 0.01) & (l2_errors < 0.05))
        poor_cases = np.sum(l2_errors >= 0.05)
        
        print(f"\\n📈 ERROR DISTRIBUTION:")
        print(f"   • Excellent (L2 < 1%): {excellent_cases}/{len(l2_errors)} ({excellent_cases/len(l2_errors)*100:.1f}%)")
        print(f"   • Good (1% ≤ L2 < 5%): {good_cases}/{len(l2_errors)} ({good_cases/len(l2_errors)*100:.1f}%)")
        print(f"   • Poor (L2 ≥ 5%): {poor_cases}/{len(l2_errors)} ({poor_cases/len(l2_errors)*100:.1f}%)")
    
    # Model complexity
    if 'total_params' in locals():
        print(f"\\n🏗️  MODEL COMPLEXITY:")
        print(f"   • Total Parameters: {total_params:,}")
        print(f"   • Trainable Parameters: {trainable_params:,}")
        print(f"   • Model Size: ~{total_params * 4 / 1024 / 1024:.2f} MB (float32)")
    
    # Performance metrics
    if 'avg_time' in locals():
        print(f"\\n⚡ COMPUTATIONAL PERFORMANCE:")
        print(f"   • Average Inference Time: {avg_time*1000:.3f} ms")
        print(f"   • Throughput: ~{1/avg_time:.1f} samples/second")
        print(f"   • Time per Sample: {avg_time*1000/test_batch_size:.3f} ms")
    
    # Training convergence
    try:
        history_data = pd.read_csv('../Code/output/Poisson_PINTO/history.csv')
        final_loss = history_data['loss'].iloc[-1]
        final_residual = history_data['residual_loss'].iloc[-1]
        final_boundary = history_data['bound_loss'].iloc[-1]
        
        print(f"\\n🎯 TRAINING CONVERGENCE:")
        print(f"   • Final Total Loss: {final_loss:.6f}")
        print(f"   • Final Residual Loss: {final_residual:.6f}")
        print(f"   • Final Boundary Loss: {final_boundary:.6f}")
        print(f"   • Total Training Epochs: {len(history_data)}")
        
        # Convergence assessment
        initial_loss = history_data['loss'].iloc[0]
        improvement_ratio = initial_loss / final_loss
        print(f"   • Loss Improvement Ratio: {improvement_ratio:.2f}x")
        
    except:
        print("\\n🎯 TRAINING CONVERGENCE: Data not available")
    
    print(f"\\n🔬 PHYSICS COMPLIANCE:")
    print(f"   • PDE: -d/dx(a(x) * dp/dx) = f(x)")
    print(f"   • Boundary Conditions: Enforced via physics loss")
    print(f"   • Context Learning: Multi-head attention for (p, a, f) relationships")
    
    print(f"\\n✅ ASSESSMENT:")
    if 'l2_errors' in locals():
        overall_performance = "Excellent" if np.mean(l2_errors) < 0.02 else \
                             "Good" if np.mean(l2_errors) < 0.05 else \
                             "Needs Improvement"
        print(f"   • Overall Model Performance: {overall_performance}")
        print(f"   • Recommended for: {'Production use' if overall_performance in ['Excellent', 'Good'] else 'Further development'}")
    
    print(f"\\n📋 RECOMMENDATIONS:")
    if 'l2_errors' in locals() and np.mean(l2_errors) > 0.05:
        print(f"   • Consider increasing model complexity or training epochs")
        print(f"   • Analyze cases with high errors for data quality issues")
        print(f"   • Experiment with different loss function weights")
    else:
        print(f"   • Model shows good performance on Poisson equation")
        print(f"   • Ready for deployment on similar problem domains")
        print(f"   • Consider testing on more complex geometries or boundary conditions")
    
    print("\\n" + "="*80)
    print("Analysis completed successfully!")
    print("Generated plots: Poisson_Solutions.png, Poisson_Error_Analysis.png, ")
    print("                Poisson_Learning_Curves.png, Poisson_Performance_Analysis.png")
    print("="*80)

except Exception as e:
    print(f"Error generating summary: {e}")
    print("Some analysis results may not be available.")
    print("\\nEnsure all previous cells have been executed successfully.")

# Save key metrics to file
try:
    if 'l2_errors' in locals() and 'max_errors' in locals():
        results_summary = {
            'mean_l2_error': np.mean(l2_errors),
            'std_l2_error': np.std(l2_errors),
            'mean_max_error': np.mean(max_errors),
            'std_max_error': np.std(max_errors),
            'min_l2_error': np.min(l2_errors),
            'max_l2_error': np.max(l2_errors),
            'total_test_cases': len(l2_errors)
        }
        
        # Save to CSV
        results_df = pd.DataFrame([results_summary])
        results_df.to_csv('Poisson_Results_Summary.csv', index=False)
        print(f"\\n💾 Results saved to: Poisson_Results_Summary.csv")
        
except Exception as e:
    print(f"Could not save results summary: {e}")